# 1. Reinforcement Learning'e Giriş

**Kaynak:** Sutton & Barto, *Reinforcement Learning: An Introduction*, 2nd Edition (2018)
- **Bölüm 1: Introduction** (Sayfa 1-16)

## İçindekiler
1. RL Nedir? *(s. 1-3)*
2. RL'nin Temel Elementleri *(s. 6-9)*
3. Agent-Environment Etkileşimi *(s. 47-48)*
4. Return ve Discounting *(s. 54-56)*

---
## 1.1 Reinforcement Learning Nedir?

📖 **Referans:** Sutton & Barto, Sayfa 1-3

> *"Reinforcement learning is learning what to do—how to map situations to actions—so as to maximize a numerical reward signal."* (s. 1)

**Reinforcement Learning (Pekiştirmeli Öğrenme)**, bir ajanın (agent) çevresiyle (environment) etkileşime girerek deneme-yanılma yoluyla öğrenmesidir.

### Diğer Öğrenme Paradigmalarından Farkı (s. 2)

| Paradigma | Özellik |
|-----------|--------|
| **Supervised Learning** | Etiketli veri, doğru cevap verilir |
| **Unsupervised Learning** | Etiketsiz veri, yapı keşfi |
| **Reinforcement Learning** | Ödül sinyali, deneme-yanılma |

📖 **Sayfa 2'den:**
> *"One of the challenges that arise in reinforcement learning, and not in other kinds of learning, is the trade-off between exploration and exploitation."*

RL'de:
- Doğru aksiyonlar **önceden verilmez**
- Agent **ödül** (reward) maksimizasyonu yapar
- **Exploration vs Exploitation** dengesi kritiktir (s. 3)

---
## 1.2 RL'nin Temel Elementleri

📖 **Referans:** Sutton & Barto, Sayfa 6-9, Section 1.3 "Elements of Reinforcement Learning"

### 1. Policy (π) - Sayfa 6
> *"A policy defines the learning agent's way of behaving at a given time."*

Agent'ın davranış stratejisi. State'den action'a mapping.

$$\pi(a|s) = P(A_t = a | S_t = s)$$

### 2. Reward Signal (R) - Sayfa 6-7
> *"The reward signal defines the goal of a reinforcement learning problem."*

Her adımda environment'ın verdiği sayısal geri bildirim.

### 3. Value Function (V) - Sayfa 7
> *"Whereas the reward signal indicates what is good in an immediate sense, a value function specifies what is good in the long run."*

Bir state'in uzun vadeli değeri.

$$V^\pi(s) = E_\pi[G_t | S_t = s]$$

### 4. Model (opsiyonel) - Sayfa 8
> *"A model of the environment... allows inferences to be made about how the environment will behave."*

Environment'ın dinamiklerinin tahmini. Model-based vs Model-free RL ayrımını belirler.

---
## 1.3 Agent-Environment Etkileşimi

📖 **Referans:** Sutton & Barto, Sayfa 47-48, Section 3.1 "The Agent-Environment Interface"

```
    +-------+     action (a)      +-----------+
    | Agent | -----------------> | Environment|
    +-------+ <----------------- +-----------+
                state (s), reward (r)
```

**Figure 3.1 (s. 48)** bu etkileşimi gösterir.

Her zaman adımında (s. 48):
1. Agent state'i gözlemler: $S_t$
2. Agent action seçer: $A_t$
3. Environment yeni state ve reward verir: $S_{t+1}, R_{t+1}$

> *"The agent and environment interact at each of a sequence of discrete time steps, t = 0, 1, 2, 3, ..."* (s. 47)

In [ ]:
# Kod Örneği: Basit Agent-Environment Loop
# Bu kod, Sutton & Barto Sayfa 48, Figure 3.1'i implement eder

import numpy as np
import matplotlib.pyplot as plt

class SimpleEnvironment:
    """
    Basit bir grid world environment.
    Referans: Sutton & Barto, Example 3.5 (s. 60) - Gridworld benzeri
    """
    
    def __init__(self, size=5):
        self.size = size
        self.goal = (size-1, size-1)  # Sağ alt köşe hedef
        self.reset()
    
    def reset(self):
        """Episode başlat, initial state döndür (s. 48)"""
        self.position = (0, 0)  # Sol üst köşeden başla
        return self.position
    
    def step(self, action):
        """
        Action al, (next_state, reward, done) döndür.
        Bu, Sayfa 48'deki dynamics'i implement eder:
        p(s', r | s, a) = Pr{S_t=s', R_t=r | S_{t-1}=s, A_{t-1}=a}
        """
        x, y = self.position
        
        # Action: 0=yukarı, 1=sağ, 2=aşağı, 3=sol
        if action == 0 and y > 0:
            y -= 1
        elif action == 1 and x < self.size - 1:
            x += 1
        elif action == 2 and y < self.size - 1:
            y += 1
        elif action == 3 and x > 0:
            x -= 1
        
        self.position = (x, y)
        
        # Reward structure (basitleştirilmiş)
        if self.position == self.goal:
            return self.position, 1.0, True
        else:
            return self.position, -0.1, False

# Test
env = SimpleEnvironment(size=5)
state = env.reset()
print(f"Başlangıç state S_0: {state}")
print(f"Hedef (terminal state): {env.goal}")

In [ ]:
# Kod Örneği: Random Policy ile Agent
# Referans: Sayfa 58 - "random policy" kavramı

class RandomAgent:
    """
    Rastgele aksiyon seçen basit bir agent.
    Bu, equiprobable random policy'dir (s. 58):
    π(a|s) = 1/|A(s)| for all a
    """
    
    def __init__(self, n_actions=4):
        self.n_actions = n_actions
    
    def select_action(self, state):
        # Her action eşit olasılıklı
        return np.random.randint(self.n_actions)

# Agent-Environment etkileşimi (Figure 3.1, s. 48)
agent = RandomAgent()
env = SimpleEnvironment(size=5)

# Bir episode çalıştır
state = env.reset()  # S_0
total_reward = 0
trajectory = [state]

for t in range(100):  # Max 100 adım
    action = agent.select_action(state)  # A_t ~ π(·|S_t)
    next_state, reward, done = env.step(action)  # S_{t+1}, R_{t+1}
    total_reward += reward
    trajectory.append(next_state)
    
    if done:
        print(f"Hedefe t={t+1} adımda ulaşıldı!")
        break
    
    state = next_state

print(f"Toplam reward (sum of R_1 to R_T): {total_reward:.2f}")

In [ ]:
def visualize_trajectory(trajectory, size=5):
    """Trajectory'yi grid üzerinde görselleştir."""
    fig, ax = plt.subplots(figsize=(6, 6))
    
    # Grid çiz
    for i in range(size + 1):
        ax.axhline(y=i, color='gray', linewidth=0.5)
        ax.axvline(x=i, color='gray', linewidth=0.5)
    
    # Trajectory çiz
    xs = [p[0] + 0.5 for p in trajectory]
    ys = [size - p[1] - 0.5 for p in trajectory]
    
    ax.plot(xs, ys, 'b-', linewidth=2, alpha=0.7)
    ax.scatter(xs[0], ys[0], color='green', s=200, zorder=5, label='Start (S_0)')
    ax.scatter(xs[-1], ys[-1], color='red', s=200, zorder=5, label='End (S_T)')
    
    # Hedef
    ax.add_patch(plt.Rectangle((size-1, 0), 1, 1, color='gold', alpha=0.5))
    ax.text(size-0.5, 0.5, 'GOAL', ha='center', va='center', fontsize=10)
    
    ax.set_xlim(0, size)
    ax.set_ylim(0, size)
    ax.set_aspect('equal')
    ax.legend()
    ax.set_title(f'Agent Trajectory ({len(trajectory)-1} steps)')
    plt.show()

visualize_trajectory(trajectory)

---
## 1.4 Return ve Discounting

📖 **Referans:** Sutton & Barto, Sayfa 54-56, Section 3.3 "Returns and Episodes"

Agent'ın amacı **expected return**'ü maksimize etmektir (s. 54):

> *"In general, we seek to maximize the expected return, where the return, denoted $G_t$, is defined as some specific function of the reward sequence."*

### Discounted Return (Equation 3.8, s. 55)

$$G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + ... = \sum_{k=0}^{\infty} \gamma^k R_{t+k+1}$$

### Discount Factor γ (s. 55-56)

> *"The discount rate determines the present value of future rewards."*

| γ değeri | Anlamı | Kitaptaki açıklama (s. 55) |
|----------|--------|----------------------------|
| γ = 0 | Myopic | "agent is 'myopic'... concerned only with maximizing immediate rewards" |
| γ → 1 | Far-sighted | "future rewards are taken into account more strongly" |
| 0 < γ < 1 | Balanced | Yakın ve uzak ödüller dengeli |

In [ ]:
# Kod Örneği: Return Hesaplama
# Referans: Equation 3.8 (s. 55)

def calculate_return(rewards, gamma=0.99):
    """
    Verilen reward listesi için discounted return hesapla.
    
    Equation 3.8 (s. 55):
    G_t = R_{t+1} + γR_{t+2} + γ²R_{t+3} + ...
    
    Recursive form (Equation 3.9, s. 55):
    G_t = R_{t+1} + γG_{t+1}
    """
    G = 0
    returns = []
    
    # Sondan başa doğru hesapla (Equation 3.9 kullanarak)
    for r in reversed(rewards):
        G = r + gamma * G  # G_t = R_{t+1} + γG_{t+1}
        returns.insert(0, G)
    
    return returns

# Örnek: Bir episode'daki reward'lar
# Senaryo: 4 adım sonra hedefe ulaşma
rewards = [-0.1, -0.1, -0.1, -0.1, 1.0]  # R_1, R_2, R_3, R_4, R_5

print("Farklı γ değerleri için G_0 (initial state'in return'ü):")
print("="*50)
for gamma in [0.0, 0.5, 0.9, 0.99, 1.0]:
    returns = calculate_return(rewards, gamma)
    print(f"γ={gamma:.2f}: G_0={returns[0]:.3f}")
    print(f"        Tüm returns: {[f'{r:.3f}' for r in returns]}")

---
## Özet

Bu notebook'ta öğrendiklerimiz (Sutton & Barto Chapter 1 & 3'ten):

| Kavram | Sayfa | Açıklama |
|--------|-------|----------|
| RL tanımı | s. 1 | Agent-environment etkileşimi ile öğrenme |
| Policy (π) | s. 6 | Agent'ın davranış stratejisi |
| Reward (R) | s. 6-7 | Anlık geri bildirim sinyali |
| Value Function (V) | s. 7 | Uzun vadeli beklenen değer |
| Model | s. 8 | Environment dinamiklerinin tahmini |
| Return (G) | s. 54-55 | Discounted kümülatif reward |
| Discount (γ) | s. 55-56 | Gelecek ödüllerin şimdiki değeri |

### Anahtar Denklemler

**Discounted Return** (Eq. 3.8, s. 55):
$$G_t = \sum_{k=0}^{\infty} \gamma^k R_{t+k+1}$$

**Recursive Return** (Eq. 3.9, s. 55):
$$G_t = R_{t+1} + \gamma G_{t+1}$$

---
### Sonraki Notebook
**02 - Multi-Armed Bandits** *(Chapter 2, s. 25-46)*: Exploration vs Exploitation problemi